# 99_comparacao_modelos

Notebook consolidado para comparar o desempenho das trilhas A, B, C, D e E em um protocolo unico de avaliacao no conjunto de teste.


## 1. Setup e objetivo

Objetivo desta secao:
- preparar ambiente reprodutivel
- definir caminhos dos artefatos de entrada e saida
- padronizar funcoes de avaliacao

Interpretacao:
- todas as trilhas serao comparadas com as mesmas metricas e o mesmo calculo
- isso reduz risco de divergencia metodologica entre notebooks individuais


In [ ]:
# ---------------------------------------------------------
# Setup reprodutivel e utilitarios de avaliacao
# ---------------------------------------------------------
# Contexto:
# O notebook consolida resultados de varios experimentos PU.
#
# Objetivo estatistico:
# Recalcular AUC_s e Recall@k com uma regra unica para todas as trilhas.
#
# Operacao computacional:
# - define paths
# - define mapeamento de trilhas
# - define funcoes auxiliares para score e metricas
#
# Interpretacao esperada:
# Ao final, teremos base padrao para comparacao justa entre modelos.

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUTPUTS_DIR = ROOT / "dados" / "dados_processados" / "outputs_pu"
if not OUTPUTS_DIR.exists():
    raise FileNotFoundError(f"Diretorio de outputs ausente: {OUTPUTS_DIR}")

COMPARISON_PARQUET = OUTPUTS_DIR / "comparacao_modelos_A_E.parquet"
COMPARISON_CSV = OUTPUTS_DIR / "comparacao_modelos_A_E.csv"

TRILHAS = [
    {
        "trilha": "A",
        "nome": "Trilha A",
        "modelo": "LogisticRegression PU",
        "paradigma": "PU (Elkan-Noto)",
        "scores_file": "trilhaA_scores_test.parquet",
    },
    {
        "trilha": "B",
        "nome": "Trilha B",
        "modelo": "SGD PU",
        "paradigma": "PU (Elkan-Noto)",
        "scores_file": "trilhaB_scores_test.parquet",
    },
    {
        "trilha": "C",
        "nome": "Trilha C",
        "modelo": "HistGradientBoosting",
        "paradigma": "Supervisionado em s (proxy)",
        "scores_file": "trilhaC_scores_test.parquet",
    },
    {
        "trilha": "D",
        "nome": "Trilha D",
        "modelo": "XGBoost",
        "paradigma": "Supervisionado em s (proxy)",
        "scores_file": "trilhaD_scores_test.parquet",
    },
    {
        "trilha": "E",
        "nome": "Trilha E",
        "modelo": "nnPU",
        "paradigma": "PU por risco nao-negativo (nnPU)",
        "scores_file": "trilhaE_scores_test.parquet",
    },
]

def pick_score_col(df: pd.DataFrame) -> str:
    # Mapeia diferentes nomes de score usados nas trilhas.
    preferred = [
        "score_corrigido",
        "score_pu_corrigido",
        "score_model",
        "score_pu",
    ]
    for col in preferred:
        if col in df.columns:
            return col

    candidates = [c for c in df.columns if "score" in c.lower()]
    if not candidates:
        raise ValueError(f"Nenhuma coluna de score encontrada. Colunas: {list(df.columns)}")
    return candidates[0]


def recall_at_fraction(y_true: np.ndarray, scores: np.ndarray, frac: float) -> tuple[int, int, float]:
    if len(y_true) != len(scores):
        raise ValueError("y_true e scores devem ter o mesmo tamanho")
    if not (0.0 < frac <= 1.0):
        raise ValueError("frac deve estar em (0, 1]")

    k = max(1, int(np.ceil(frac * len(y_true))))
    top_idx = np.argsort(-scores)[:k]
    total_pos = int((y_true == 1).sum())
    hit_pos = int((y_true[top_idx] == 1).sum())
    recall_k = hit_pos / total_pos if total_pos > 0 else np.nan
    return k, hit_pos, float(recall_k)


print(f"Diretorio de outputs: {OUTPUTS_DIR}")
print(f"Arquivo final parquet: {COMPARISON_PARQUET.name}")
print(f"Arquivo final csv: {COMPARISON_CSV.name}")


## 2. Descoberta dos artefatos A-E

Nesta secao, validamos disponibilidade dos arquivos `scores_test` por trilha.

Interpretacao:
- se faltar artefato de uma trilha, a comparacao fica parcial
- o notebook registra de forma explicita qualquer ausencia


In [ ]:
discovery_rows = []
for item in TRILHAS:
    path = OUTPUTS_DIR / item["scores_file"]
    discovery_rows.append(
        {
            "trilha": item["trilha"],
            "arquivo_scores": item["scores_file"],
            "existe": path.exists(),
            "path": str(path),
        }
    )

discovery_df = pd.DataFrame(discovery_rows).sort_values("trilha").reset_index(drop=True)
display(discovery_df)

if not discovery_df["existe"].all():
    missing = discovery_df.loc[~discovery_df["existe"], ["trilha", "arquivo_scores"]]
    raise FileNotFoundError(f"Artefatos ausentes para comparacao: {missing.to_dict(orient='records')}")


## 3. Recalculo unificado das metricas no teste

Esta secao recalcula todas as metricas diretamente dos `scores_test` para garantir comparacao homogenea:
- `AUC_s`
- `Recall@1%`
- `Recall@5%`
- `Recall@10%`

Observacao metodologica:
- as metricas sao calculadas em `s` observado (proxy)
- elas nao medem diretamente `y` latente


In [ ]:
metrics_rows = []

for item in TRILHAS:
    scores_path = OUTPUTS_DIR / item["scores_file"]
    df_scores = pd.read_parquet(scores_path, engine="pyarrow")

    if "s" not in df_scores.columns:
        raise ValueError(f"Coluna 's' ausente em {scores_path.name}")

    score_col = pick_score_col(df_scores)
    y_true = pd.to_numeric(df_scores["s"], errors="coerce").fillna(0).astype("int8").to_numpy()
    scores = pd.to_numeric(df_scores[score_col], errors="coerce").fillna(0.0).astype("float64").to_numpy()

    auc_s = float(roc_auc_score(y_true, scores))
    _, _, r1 = recall_at_fraction(y_true, scores, 0.01)
    _, _, r5 = recall_at_fraction(y_true, scores, 0.05)
    _, _, r10 = recall_at_fraction(y_true, scores, 0.10)

    metrics_rows.append(
        {
            "trilha": item["trilha"],
            "nome_trilha": item["nome"],
            "modelo": item["modelo"],
            "paradigma": item["paradigma"],
            "arquivo_scores": scores_path.name,
            "score_col": score_col,
            "n_test": int(len(df_scores)),
            "AUC_s": auc_s,
            "Recall@1%": float(r1),
            "Recall@5%": float(r5),
            "Recall@10%": float(r10),
        }
    )

metrics_df = pd.DataFrame(metrics_rows).sort_values("trilha").reset_index(drop=True)
display(metrics_df)


## 4. Comparacao final e leitura executiva

Nesta secao:
- ordenamos as trilhas por `AUC_s`
- calculamos delta vs baseline (Trilha A)
- calculamos delta vs melhor trilha anterior

Interpretacao:
- a comparacao suporta decisao operacional e narrativa do artigo
- a leitura deve considerar o paradigma de cada trilha


In [ ]:
baseline_a = metrics_df.loc[metrics_df["trilha"] == "A"].iloc[0]

comparison_df = metrics_df.copy()
comparison_df["rank_auc_s"] = comparison_df["AUC_s"].rank(method="dense", ascending=False).astype("int64")

comparison_df["delta_auc_vs_A"] = comparison_df["AUC_s"] - float(baseline_a["AUC_s"])
comparison_df["delta_r1_vs_A"] = comparison_df["Recall@1%"] - float(baseline_a["Recall@1%"])
comparison_df["delta_r5_vs_A"] = comparison_df["Recall@5%"] - float(baseline_a["Recall@5%"])
comparison_df["delta_r10_vs_A"] = comparison_df["Recall@10%"] - float(baseline_a["Recall@10%"])

best_prev_auc = comparison_df.loc[comparison_df["trilha"].isin(["A", "B", "C", "D"]), "AUC_s"].max()
comparison_df["delta_auc_vs_best_prev"] = comparison_df["AUC_s"] - float(best_prev_auc)

comparison_df = comparison_df.sort_values(["AUC_s", "Recall@10%"], ascending=[False, False]).reset_index(drop=True)
display(comparison_df)

print("Ranking por AUC_s:")
display(comparison_df[["trilha", "modelo", "AUC_s", "Recall@1%", "Recall@5%", "Recall@10%", "rank_auc_s"]])


## 5. Persistencia de artefatos consolidados

Persistimos dois formatos:
- parquet para pipeline
- csv para leitura rapida e compartilhamento

Isso torna a comparacao reutilizavel em outros notebooks e no texto do artigo.


In [ ]:
comparison_df.to_parquet(COMPARISON_PARQUET, index=False)
comparison_df.to_csv(COMPARISON_CSV, index=False)

print(f"Parquet salvo: {COMPARISON_PARQUET}")
print(f"CSV salvo: {COMPARISON_CSV}")


## 6. Conclusao operacional

Resumo rapido para decisao:
- melhor trilha por `AUC_s`
- melhor trilha por `Recall@10%`
- posicao relativa da trilha E frente as trilhas supervisionadas em `s`


In [ ]:
best_auc_row = comparison_df.sort_values("AUC_s", ascending=False).iloc[0]
best_r10_row = comparison_df.sort_values("Recall@10%", ascending=False).iloc[0]
row_e = comparison_df.loc[comparison_df["trilha"] == "E"].iloc[0]

print("Resumo final da comparacao A-E")
print(f"- Melhor AUC_s: Trilha {best_auc_row['trilha']} ({best_auc_row['AUC_s']:.6f})")
print(f"- Melhor Recall@10%: Trilha {best_r10_row['trilha']} ({best_r10_row['Recall@10%']:.6f})")
print(f"- Trilha E AUC_s: {row_e['AUC_s']:.6f} | Recall@10%: {row_e['Recall@10%']:.6f}")
